In [0]:
# Catalog Name
catalog = "workspace"

# Source Schema
source_schema = "silvers"

# Source Object
source_object = "silver_bookings"

# CDC Column
cdc_col = "updated_timestamp"   

# Backdated Refresh
backdated_refresh = ""

# Source Fact Table
fact_table = f"{catalog}.{source_schema}.{source_object}"

# Target Schema
target_schema = "golds"

# Target Object
target_object = "fact_bookings"

# Fact Key
key_cols_list = ["booking_id"]   # ← natural key for fact table

In [0]:
dimensions = [
    {
        "table": f"{catalog}.{target_schema}.dim_passengers",
        "alias": "dim_passengers",
        "join_keys": [("passenger_id", "passenger_id")],
        "surrogate_key": "Dimpassengerkey"   # (fact_col, dim_col)
    },
    {
        "table": f"{catalog}.{target_schema}.dim_flights",
        "alias": "dim_flights",
        "join_keys": [("flight_id", "flight_id")],
        "surrogate_key": "Dimflightkey"   # (fact_col, dim_col)
    },
    {
        "table": f"{catalog}.{target_schema}.dim_airports",
        "alias": "dim_aiports",
        "join_keys": [("airport_id", "airport_id")],
        "surrogate_key": "Dimairportkey"  # (fact_col, dim_col)
    },
]




# Columns you want to keep from Fact table (besides the surrogate keys)
fact_columns = ["booking_id", "amount", "booking_date"]

In [0]:
# No Back Dated Refresh
if len(backdated_refresh) == 0:
  
  # If Table Exists In The Destination
  if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

    last_load = spark.sql(f"SELECT max({cdc_column}) FROM workspace.{target_schema}.{target_object}").collect()[0][0]
    
  else:
    last_load = "1900-01-01 00:00:00"

# Yes Back Dated Refresh
else:
  last_load = backdated_refresh

# Test The Last Load 
last_load

'1900-01-01 00:00:00'

DYNAMIC FACT QUERY [BRING KEYS]

In [0]:
def generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_col, last_load):
    fact_alias = "f"
    
    select_cols = [f"{fact_alias}.{col}" for col in fact_columns]

    join_clauses = []
    for dim in dimensions:
        table_full = dim["table"]
        alias = dim["alias"]
        surrogate_key = f"{alias}.{dim['surrogate_key']}"  # ← fixed
        select_cols.append(surrogate_key)

        on_conditions = [
            f"{fact_alias}.{fk} = {alias}.{dk}" for fk, dk in dim["join_keys"]
        ]
        join_clause = f"LEFT JOIN {table_full} {alias} ON " + " AND ".join(on_conditions)
        join_clauses.append(join_clause)

    select_clause = ",\n    ".join(select_cols)
    joins = "\n".join(join_clauses)
    where_clause = f"{fact_alias}.{cdc_col} >= DATE('{last_load}')"

    query = f"""
SELECT
    {select_clause}
FROM {fact_table} {fact_alias}
{joins}
WHERE {where_clause}
""".strip()

    return query

In [0]:
query = generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_col, last_load)
print(query)   # ← always print to verify generated SQL before running it

SELECT
    f.booking_id,
    f.amount,
    f.booking_date,
    dim_passengers.Dimpassengerkey,
    dim_flights.Dimflightkey,
    dim_aiports.Dimairportkey
FROM workspace.silvers.silver_bookings f
LEFT JOIN workspace.golds.dim_passengers dim_passengers ON f.passenger_id = dim_passengers.passenger_id
LEFT JOIN workspace.golds.dim_flights dim_flights ON f.flight_id = dim_flights.flight_id
LEFT JOIN workspace.golds.dim_airports dim_aiports ON f.airport_id = dim_aiports.airport_id
WHERE f.updated_timestamp >= DATE('1900-01-01 00:00:00')


## DF_FACT

In [0]:
df_fact = spark.sql(query)

UPSERT

In [0]:
# Fact Key Columns Merge Condition
fact_key_cols_str = " AND ".join([f"src.{col} = trg.{col}" for col in fact_key_cols])
fact_key_cols_str

'src.DimPassengersKey = trg.DimPassengersKey AND src.DimFlightsKey = trg.DimFlightsKey AND src.DimAirportsKey = trg.DimAirportsKey AND src.booking_date = trg.booking_date'

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

    dlt_obj = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
    dlt_obj.alias("trg").merge(df_fact.alias("src"), fact_key_cols_str)\
                        .whenMatchedUpdateAll(condition = f"src.{cdc_column} >= trg.{cdc_column}")\
                        .whenNotMatchedInsertAll()\
                        .execute()

else: 

    df_fact.write.format("delta")\
            .mode("append")\
            .saveAsTable(f"{catalog}.{target_schema}.{target_object}")


In [0]:
%sql
SELECT * FROM workspace.golds.fact_bookings

booking_id,amount,booking_date,Dimpassengerkey,Dimflightkey,Dimairportkey
B01001,912.64,2025-07-10,207,47,4
B01002,597.04,2025-07-01,124,88,51
B01003,724.88,2025-07-10,103,96,36
B01004,1370.99,2025-07-12,49,69,1
B01005,1476.15,2025-07-01,210,84,53
B01006,800.39,2025-06-26,126,84,42
B01007,1187.22,2025-06-28,77,13,55
B01008,913.43,2025-07-18,118,5,33
B01009,260.75,2025-07-08,90,102,34
B01010,245.61,2025-07-21,12,80,45
